In [144]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

import warnings
warnings.simplefilter("ignore")

In [145]:
X_train = pd.read_csv("data/train.csv")
y_train = X_train["Survived"]
X_test = pd.read_csv("data/test.csv")
y_test = pd.read_csv("data/gender_submission.csv")["Survived"]
ids = X_test["PassengerId"]

In [146]:
X_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [147]:
# gender OneHotEncoding
ohe = OneHotEncoder(sparse_output=False, drop='first')
encoded = ohe.fit_transform(X_train[['Sex']])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
X_train = X_train.drop(columns=['Sex'])
X_train = pd.concat([X_train, encoded_df], axis=1)

encoded = ohe.transform(X_test[['Sex']])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
X_test = X_test.drop(columns=['Sex'])
X_test = pd.concat([X_test, encoded_df], axis=1)

In [148]:
# feature engineering
X_train["family"] = X_train["SibSp"] + X_train["Parch"]
X_train = X_train.drop(["SibSp", "Parch"], axis=1)

X_test["family"] = X_test["SibSp"] + X_test["Parch"]
X_test = X_test.drop(["SibSp", "Parch"], axis=1)

In [149]:
# fill Age NaN values
imputer = SimpleImputer(strategy='median')
X_train[['Age', 'Fare']] = imputer.fit_transform(X_train[['Age', 'Fare']])

X_test[['Age', 'Fare']] = imputer.transform(X_test[['Age', 'Fare']])

In [150]:
# Cabin
X_train["haveCabin"] = np.where(X_train["Cabin"].isnull(), 0, 1)
X_train = X_train.drop(["Cabin"], axis=1)

X_test["haveCabin"] = np.where(X_test["Cabin"].isnull(), 0, 1)
X_test = X_test.drop(["Cabin"], axis=1)

In [151]:
# drop
del_features = ["PassengerId", "Name", "Ticket", "Embarked"]
X_train = X_train.drop(del_features, axis=1)
X_train = X_train.drop(["Survived"], axis=1)
X_test = X_test.drop(del_features, axis=1)

In [152]:
train_data.head()

,Pclass,Age,Fare,Sex_male,family,haveCabin
0,3,22.0,7.2500,1.0,1,0
1,1,38.0,71.2833,0.0,1,1
2,3,26.0,7.9250,0.0,0,0
3,1,35.0,53.1000,0.0,1,1
4,3,35.0,8.0500,1.0,0,0


In [153]:
# corr_mx = X_train.corr()
# corr_mx = X_train.corr(numeric_only=True)
# sns.heatmap(corr_mx, annot=True, fmt=".2f", cmap="coolwarm")

## models

In [154]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print("----- Random Rorest -----")
print(f"acc: {round(rf_acc, 5)}")

----- Random Rorest -----
acc: 0.82057


In [155]:
submission = pd.DataFrame({
    'PassengerId': ids,
    'Survived': rf_pred
})

submission.to_csv('submission.csv', index=False)